# Territorial Digital Divide in Cusco

            This notebook measures territorial digital divide patterns in Cusco, Peru, by combining NASA nighttime lights with OSIPTEL mobile coverage density.

            Required local inputs:

            - `data/VNL_cusco_2025.tif`
            - `data/kernel_cobmovil2019_50m.tif`

## Step 0 - Environment setup

            Import required packages and print versions for reproducibility.

In [ ]:
from pathlib import Path
            import math
            import warnings

            import numpy as np
            import pandas as pd
            import rasterio
            from rasterio.enums import Resampling
            from rasterio.warp import reproject
            import matplotlib
            import matplotlib.pyplot as plt
            from matplotlib.colors import ListedColormap, BoundaryNorm
            from matplotlib.patches import Patch
            import scipy
            from scipy import ndimage, stats
            import seaborn as sns

            warnings.filterwarnings("ignore", category=RuntimeWarning)
            sns.set_theme(style="whitegrid")

            cwd = Path.cwd()
            if (cwd / "data").exists():
                PROJECT_ROOT = cwd
            elif (cwd.parent / "data").exists():
                PROJECT_ROOT = cwd.parent
            else:
                PROJECT_ROOT = cwd
            DATA_DIR = PROJECT_ROOT / "data"
            OUTPUT_DIR = PROJECT_ROOT / "output"
            OUTPUT_DIR.mkdir(exist_ok=True)

            VNL_PATH = DATA_DIR / "VNL_cusco_2025.tif"
            CONN_PATH = DATA_DIR / "kernel_cobmovil2019_50m.tif"

            versions = {
                "rasterio": rasterio.__version__,
                "numpy": np.__version__,
                "matplotlib": matplotlib.__version__,
                "scipy": scipy.__version__,
                "seaborn": sns.__version__,
                "pandas": pd.__version__,
            }
            versions

## Step 1 - Raster loading and inspection

            For each raster, print CRS, dimensions, band count, NoData value, data type, bounds, pixel resolution, valid pixels, and value range.

In [ ]:
def pixel_resolution_report(src):
                xres, yres = abs(src.transform.a), abs(src.transform.e)
                bounds = src.bounds
                center_lat = (bounds.bottom + bounds.top) / 2

                if src.crs and src.crs.is_geographic:
                    x_deg, y_deg = xres, yres
                    x_km = x_deg * 111.32 * math.cos(math.radians(center_lat))
                    y_km = y_deg * 110.57
                    return {
                        "resolution_degrees": (x_deg, y_deg),
                        "approx_resolution_km": (abs(x_km), abs(y_km)),
                    }

                x_km = xres / 1000
                y_km = yres / 1000
                x_deg = xres / (111_320 * max(math.cos(math.radians(center_lat)), 0.01))
                y_deg = yres / 110_570
                return {
                    "resolution_degrees_approx": (abs(x_deg), abs(y_deg)),
                    "resolution_km": (abs(x_km), abs(y_km)),
                }


            def read_clean_array(path):
                with rasterio.open(path) as src:
                    arr = src.read(1).astype("float32")
                    nodata = src.nodata
                    mask = np.zeros(arr.shape, dtype=bool)
                    if nodata is not None:
                        mask |= arr == nodata
                    mask |= ~np.isfinite(arr)
                    clean = arr.copy()
                    clean[mask] = np.nan
                    return clean, src.profile.copy()


            def inspect_raster(path, label):
                with rasterio.open(path) as src:
                    arr = src.read(1).astype("float32")
                    nodata = src.nodata
                    valid = np.isfinite(arr)
                    if nodata is not None:
                        valid &= arr != nodata
                    values = arr[valid]
                    report = {
                        "label": label,
                        "path": str(path),
                        "crs": str(src.crs),
                        "shape_height_width": (src.height, src.width),
                        "band_count": src.count,
                        "nodata": nodata,
                        "dtype": src.dtypes[0],
                        "bounds": src.bounds,
                        "pixel_resolution": pixel_resolution_report(src),
                        "valid_pixels": int(values.size),
                        "value_min": float(np.nanmin(values)),
                        "value_max": float(np.nanmax(values)),
                    }
                print(f"\n--- {label} ---")
                for key, value in report.items():
                    print(f"{key}: {value}")
                return report


            vnl_report = inspect_raster(VNL_PATH, "VNL nighttime lights")
            conn_report = inspect_raster(CONN_PATH, "OSIPTEL mobile coverage")

## Step 2 - Reprojection and grid alignment

            The connectivity raster is reprojected from EPSG:32719 to the exact VNL grid in EPSG:4326 using bilinear resampling.

In [ ]:
vnl_raw, vnl_profile = read_clean_array(VNL_PATH)

            with rasterio.open(VNL_PATH) as vnl_src, rasterio.open(CONN_PATH) as conn_src:
                conn_aligned = np.zeros((vnl_src.height, vnl_src.width), dtype="float32")
                source = conn_src.read(1).astype("float32")
                if conn_src.nodata is not None:
                    source[source == conn_src.nodata] = np.nan
                source[~np.isfinite(source)] = np.nan

                reproject(
                    source=source,
                    destination=conn_aligned,
                    src_transform=conn_src.transform,
                    src_crs=conn_src.crs,
                    src_nodata=np.nan,
                    dst_transform=vnl_src.transform,
                    dst_crs=vnl_src.crs,
                    dst_nodata=np.nan,
                    resampling=Resampling.bilinear,
                )

            print("VNL shape:", vnl_raw.shape)
            print("Aligned connectivity shape:", conn_aligned.shape)
            assert vnl_raw.shape == conn_aligned.shape
            print("Grid alignment verified: both rasters share identical dimensions.")

## Step 3 - Robust normalization

            Normalize both rasters with a 2nd-98th percentile scaling. Negative values, NoData, and non-finite values are replaced with zero before clipping.

In [ ]:
def robust_normalize(arr, label):
                clean = arr.astype("float32").copy()
                clean[~np.isfinite(clean)] = 0
                clean[clean < 0] = 0
                p2, p98 = np.percentile(clean[clean > 0], [2, 98]) if np.any(clean > 0) else (0, 1)
                norm = (clean - p2) / (p98 - p2) if p98 > p2 else clean * 0
                norm = np.clip(norm, 0, 1).astype("float32")
                print(
                    f"{label}: min={norm.min():.4f}, max={norm.max():.4f}, "
                    f"mean={norm.mean():.4f}, std={norm.std():.4f}, p2={p2:.4f}, p98={p98:.4f}"
                )
                return norm


            vnl_norm = robust_normalize(vnl_raw, "VNL normalized")
            conn_norm = robust_normalize(conn_aligned, "Connectivity normalized")

## Step 4 - Map 1: VNL nighttime lights

            Raw and normalized nighttime lights. Bright zones indicate urbanization and economic activity concentration.

In [ ]:
def raster_extent(profile):
                transform = profile["transform"]
                height, width = profile["height"], profile["width"]
                left, top = transform * (0, 0)
                right, bottom = transform * (width, height)
                return [left, right, bottom, top]


            extent = raster_extent(vnl_profile)

            fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
            im0 = axes[0].imshow(np.nan_to_num(vnl_raw, nan=0), cmap="inferno", extent=extent)
            axes[0].set_title("Raw VNL nighttime lights")
            axes[0].set_xlabel("Longitude")
            axes[0].set_ylabel("Latitude")
            fig.colorbar(im0, ax=axes[0], shrink=0.85)

            im1 = axes[1].imshow(vnl_norm, cmap="inferno", extent=extent, vmin=0, vmax=1)
            axes[1].set_title("Normalized VNL nighttime lights")
            axes[1].set_xlabel("Longitude")
            axes[1].set_ylabel("Latitude")
            fig.colorbar(im1, ax=axes[1], shrink=0.85)

            plt.show()
            print("Observation: the brightest zones identify urban and economically active areas, used here as a proxy for population concentration.")

## Step 5 - Map 2: Digital Divide Index and Total Digital Exclusion

            IBD captures areas with more light than connectivity. EDT captures areas with neither light nor connectivity.

In [ ]:
ibd = (vnl_norm - conn_norm).astype("float32")
            edt = ((1 - vnl_norm) * (1 - conn_norm)).astype("float32")

            fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
            im0 = axes[0].imshow(ibd, cmap="RdYlGn_r", extent=extent, vmin=-1, vmax=1)
            axes[0].set_title("IBD: Digital Divide Index")
            axes[0].set_xlabel("Longitude")
            axes[0].set_ylabel("Latitude")
            fig.colorbar(im0, ax=axes[0], shrink=0.85)

            im1 = axes[1].imshow(edt, cmap="Purples", extent=extent, vmin=0, vmax=1)
            axes[1].set_title("EDT: Total Digital Exclusion")
            axes[1].set_xlabel("Longitude")
            axes[1].set_ylabel("Latitude")
            fig.colorbar(im1, ax=axes[1], shrink=0.85)
            plt.show()

            print("IBD interpretation: red areas have higher nighttime light than connectivity and indicate active digital divide.")
            print("EDT interpretation: darker purple areas have both low nighttime light and low connectivity, indicating total exclusion.")

## Step 6 - Map 3: Intervention priority

            Priority levels combine population proxy intensity and low connectivity.

In [ ]:
priority = np.zeros(vnl_norm.shape, dtype="uint8")
            priority[(vnl_norm >= 0.10) & (conn_norm < 0.25)] = 1
            priority[(vnl_norm >= 0.15) & (conn_norm < 0.15)] = 2
            priority[(vnl_norm >= 0.30) & (conn_norm < 0.10)] = 3

            priority_cmap = ListedColormap(["#f7f7f7", "#ffd166", "#f77f00", "#d62828"])
            priority_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], priority_cmap.N)

            fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
            ax.imshow(priority, cmap=priority_cmap, norm=priority_norm, extent=extent)
            ax.set_title("Intervention priority")
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
            ax.legend(
                handles=[
                    Patch(color="#ffd166", label="P1 Medium"),
                    Patch(color="#f77f00", label="P2 High"),
                    Patch(color="#d62828", label="P3 Critical"),
                ],
                loc="lower left",
            )
            plt.show()

            total_pixels = priority.size
            for level, name in [(1, "P1 Medium"), (2, "P2 High"), (3, "P3 Critical")]:
                count = int((priority == level).sum())
                print(f"{name}: {count:,} pixels ({count / total_pixels * 100:.2f}% of total area)")

## Step 7 - Map 4: Social Exclusion Risk

            Risk is based on total exclusion and low light, then smoothed with a Gaussian filter to show regional patterns.

In [ ]:
risk_raw = edt * (1 - vnl_norm)
            risk_raw = robust_normalize(risk_raw, "Social exclusion risk")
            risk_smooth = ndimage.gaussian_filter(risk_raw, sigma=5)

            p75, p90 = np.percentile(risk_raw, [75, 90])

            fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
            im0 = axes[0].imshow(risk_raw, cmap="hot_r", extent=extent, vmin=0, vmax=1)
            axes[0].set_title("Raw social exclusion risk")
            axes[0].set_xlabel("Longitude")
            axes[0].set_ylabel("Latitude")
            fig.colorbar(im0, ax=axes[0], shrink=0.85)

            im1 = axes[1].imshow(risk_smooth, cmap="hot_r", extent=extent, vmin=0, vmax=1)
            axes[1].set_title("Smoothed social exclusion risk")
            axes[1].set_xlabel("Longitude")
            axes[1].set_ylabel("Latitude")
            fig.colorbar(im1, ax=axes[1], shrink=0.85)
            plt.show()

            print(f"Risk threshold p75: {p75:.4f}")
            print(f"Risk threshold p90: {p90:.4f}")

## Step 8 - Territorial classification

            Apply a 2 by 2 classification using thresholds VNL >= 0.15 and Connectivity >= 0.15.

In [ ]:
classification = np.zeros(vnl_norm.shape, dtype="uint8")
            urban = vnl_norm >= 0.15
            connected = conn_norm >= 0.15
            classification[urban & connected] = 1
            classification[urban & ~connected] = 2
            classification[~urban & connected] = 3
            classification[~urban & ~connected] = 4

            class_info = {
                1: {"name": "Urban Connected", "color": "#2ca25f"},
                2: {"name": "Urban Divide", "color": "#de2d26"},
                3: {"name": "Rural Connected", "color": "#3182bd"},
                4: {"name": "Critical Divide", "color": "#756bb1"},
            }
            class_cmap = ListedColormap([class_info[i]["color"] for i in [1, 2, 3, 4]])
            class_norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5], class_cmap.N)

            fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
            ax.imshow(classification, cmap=class_cmap, norm=class_norm, extent=extent)
            ax.set_title("Territorial digital divide classification")
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
            ax.legend(
                handles=[Patch(color=class_info[i]["color"], label=class_info[i]["name"]) for i in [1, 2, 3, 4]],
                loc="lower left",
            )
            plt.show()

            pixel_width_deg = abs(vnl_profile["transform"].a)
            pixel_height_deg = abs(vnl_profile["transform"].e)
            mean_lat = (vnl_profile["transform"].f + (vnl_profile["transform"].f + vnl_profile["height"] * vnl_profile["transform"].e)) / 2
            pixel_area_km2 = (pixel_width_deg * 111.32 * math.cos(math.radians(mean_lat))) * (pixel_height_deg * 110.57)

            class_rows = []
            for class_id in [1, 2, 3, 4]:
                count = int((classification == class_id).sum())
                class_rows.append(
                    {
                        "class": class_id,
                        "name": class_info[class_id]["name"],
                        "pixel_count": count,
                        "percentage_total_area": count / classification.size * 100,
                        "approx_area_km2": count * pixel_area_km2,
                    }
                )

            class_df = pd.DataFrame(class_rows)
            display(class_df)

## Step 9 - Statistical summary

            Compute grouped descriptive statistics, Pearson correlation, KDE distributions, Welch t-test, and Cohen's d.

In [ ]:
sample_step = 40
            flat_df = pd.DataFrame(
                {
                    "vnl": vnl_norm.ravel(),
                    "connectivity": conn_norm.ravel(),
                    "ibd": ibd.ravel(),
                    "class": classification.ravel(),
                }
            )
            flat_df["class_name"] = flat_df["class"].map({k: v["name"] for k, v in class_info.items()})

            summary = flat_df.groupby("class_name")[["vnl", "connectivity", "ibd"]].agg(["mean", "std", "min", "max"])
            display(summary)

            sample_vnl = vnl_norm[::sample_step, ::sample_step].ravel()
            sample_conn = conn_norm[::sample_step, ::sample_step].ravel()
            pearson_r, pearson_p = stats.pearsonr(sample_vnl, sample_conn)
            print(f"Pearson correlation, every {sample_step}th pixel: r={pearson_r:.4f}, p={pearson_p:.4g}")

            kde_sample = flat_df.iloc[::sample_step].copy()
            kde_long = kde_sample.melt(
                id_vars=["class_name"],
                value_vars=["vnl", "connectivity"],
                var_name="layer",
                value_name="value",
            )
            plt.figure(figsize=(10, 6))
            sns.kdeplot(data=kde_long, x="value", hue="class_name", style="layer", common_norm=False)
            plt.title("KDE distributions by class")
            plt.xlabel("Normalized value")
            plt.ylabel("Density")
            plt.show()

            class1_vnl = flat_df.loc[flat_df["class"] == 1, "vnl"].iloc[::sample_step]
            class4_vnl = flat_df.loc[flat_df["class"] == 4, "vnl"].iloc[::sample_step]
            t_stat, p_value = stats.ttest_ind(class1_vnl, class4_vnl, equal_var=False)
            pooled_sd = math.sqrt(((class1_vnl.std(ddof=1) ** 2) + (class4_vnl.std(ddof=1) ** 2)) / 2)
            cohen_d = (class1_vnl.mean() - class4_vnl.mean()) / pooled_sd if pooled_sd else np.nan

            print(f"Welch t-test Class 1 vs Class 4 VNL: t={t_stat:.4f}, p={p_value:.4g}, Cohen d={cohen_d:.4f}")

## Step 10 - Export deliverables

            Export processed rasters aligned to the VNL grid and save the final dashboard figure.

In [ ]:
def export_raster(path, array, dtype="float32", nodata=None):
                profile = vnl_profile.copy()
                profile.update(
                    driver="GTiff",
                    count=1,
                    dtype=dtype,
                    compress="lzw",
                    nodata=nodata,
                )
                with rasterio.open(path, "w", **profile) as dst:
                    dst.write(array.astype(dtype), 1)
                print(f"Saved {path}")


            export_raster(OUTPUT_DIR / "vnl_norm.tif", vnl_norm, "float32")
            export_raster(OUTPUT_DIR / "conn_norm.tif", conn_norm, "float32")
            export_raster(OUTPUT_DIR / "ibd_brecha_digital.tif", ibd, "float32")
            export_raster(OUTPUT_DIR / "clasificacion_brecha.tif", classification, "uint8", nodata=0)

            fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
            panels = [
                (vnl_norm, "VNL normalized", "inferno", 0, 1),
                (conn_norm, "Connectivity normalized", "viridis", 0, 1),
                (ibd, "IBD digital divide", "RdYlGn_r", -1, 1),
                (edt, "EDT total exclusion", "Purples", 0, 1),
                (priority, "Intervention priority", priority_cmap, None, None),
                (risk_smooth, "Smoothed exclusion risk", "hot_r", 0, 1),
            ]
            for ax, (arr, title, cmap, vmin, vmax) in zip(axes.ravel(), panels):
                im = ax.imshow(arr, cmap=cmap, extent=extent, vmin=vmin, vmax=vmax)
                ax.set_title(title)
                ax.set_xlabel("Longitude")
                ax.set_ylabel("Latitude")
                fig.colorbar(im, ax=ax, shrink=0.75)
            dashboard_path = OUTPUT_DIR / "dashboard_brecha_digital.png"
            fig.savefig(dashboard_path, dpi=150)
            plt.show()
            print(f"Saved {dashboard_path}")